# Revisión base SIDCAR

Este notebook **no ejecuta Playwright dentro del kernel de Jupyter**. En su lugar, abre una **terminal externa** y lanza el script extractor (`script_sidcar_export_directo.py`), que es la forma estable en Windows.

Flujo:
1. Ejecuta la celda **Lanzar extractor**.
2. Se abre una terminal externa.
3. Se abre SIDCAR para login o reutiliza sesión.
4. El script entra al reporte, ejecuta búsqueda, exporta y guarda el archivo.
5. Vuelves al notebook y ejecutas la celda **Cargar último Excel**.


In [ ]:
from pathlib import Path
import pandas as pd
import subprocess
import sys
import os


In [ ]:
# =========================
# RUTAS DEL PROYECTO
# =========================
PROJECT_DIR = Path(r"C:\Users\JULIANCHO ROBLES\Desktop\PROYECTOS GITHUB\oficios_notificaciones")
BASES_DIR = PROJECT_DIR / "bases_actualizadas"
SCRIPT_EXTRACTOR = PROJECT_DIR / "script_sidcar_export_directo.py"

print('Proyecto:', PROJECT_DIR)
print('Bases:', BASES_DIR)
print('Script extractor:', SCRIPT_EXTRACTOR)

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del proyecto: {PROJECT_DIR}')

if not BASES_DIR.exists():
    BASES_DIR.mkdir(parents=True, exist_ok=True)

if not SCRIPT_EXTRACTOR.exists():
    raise FileNotFoundError(
        f'No existe el script extractor: {SCRIPT_EXTRACTOR}\n'
        'Guárdalo en la raíz del proyecto antes de continuar.'
    )


## Lanzar extractor en terminal externa

Esta celda abre una **ventana CMD nueva** y ejecuta `script_sidcar_export_directo.py`.

Ahí es donde harás el login si SIDCAR lo pide, y el script te llevará al reporte para exportar la base.


In [ ]:
# =========================
# LANZAR EXTRACTOR EN TERMINAL EXTERNA
# =========================
python_exe = sys.executable
comando = f'cd /d "{PROJECT_DIR}" && "{python_exe}" "{SCRIPT_EXTRACTOR.name}"'

subprocess.Popen([
    'cmd', '/k', comando
], cwd=str(PROJECT_DIR))

print('✅ Se abrió una terminal externa con el extractor de SIDCAR.')
print('Cuando termine la descarga, vuelve al notebook y ejecuta la celda de carga.')


## Cargar último Excel descargado

Ejecuta esta celda **después** de que el extractor termine y haya guardado el archivo `radicados_sidcar_*.xlsx`.


In [ ]:
# =========================
# CARGAR ÚLTIMO EXCEL VÁLIDO DE SIDCAR
# =========================
excel_files = sorted(
    BASES_DIR.glob('radicados_sidcar_*.xlsx'),
    key=lambda x: x.stat().st_mtime,
    reverse=True
)

if not excel_files:
    raise FileNotFoundError(
        'No se encontró ningún archivo radicados_sidcar_*.xlsx en bases_actualizadas.\n'
        'Primero ejecuta la celda de extractor y confirma que la descarga terminó.'
    )

ruta_excel = excel_files[0]
print('Excel encontrado:', ruta_excel)

df = pd.read_excel(ruta_excel)
print('Filas:', len(df))
print('Columnas:', len(df.columns))
display(df.head())


In [ ]:
# =========================
# VER COLUMNAS
# =========================
df.columns.tolist()


In [ ]:
# =========================
# VALIDACIÓN SIMPLE: ASEGURAR QUE NO SON COOKIES
# =========================
columnas = [str(c).lower() for c in df.columns]

if {'name', 'value', 'domain', 'path'}.issubset(set(columnas)):
    raise ValueError('El archivo cargado corresponde a sesión/cookies, no al reporte de SIDCAR.')

print('✅ La base cargada no parece ser de sesión/cookies.')
